# 05 - Model comparison

Load the saved metrics and compare the seven models on every metric, not just accuracy.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)

In [2]:
from app.ml.evaluate import load_metrics, results_to_frame

metrics = load_metrics()
if not metrics:
    raise SystemExit("No metrics found - run notebook 04 or 'python -m app.ml.train' first.")
comparison = results_to_frame(metrics)
comparison

,model,display_name,accuracy,precision,recall,f1_score,specificity,false_positive_rate,false_negative_rate,roc_auc,pr_auc,training_time,prediction_time
0,logistic_regression,Logistic Regression,0.908539,0.904561,0.895761,0.900140,0.919431,0.080569,0.104239,0.970647,0.970106,0.059463,0.000617
1,svm,SVM,0.907899,0.899931,0.899931,0.899931,0.914692,0.085308,0.100069,0.967956,0.966719,18.727430,1.481969
2,xgboost,XGBoost,0.907899,0.901045,0.898541,0.899791,0.915877,0.084123,0.101459,0.966899,0.966888,1.219319,0.033048
3,ann,ANN,0.909818,0.920727,0.879778,0.899787,0.935427,0.064573,0.120222,0.969485,0.969029,0.841196,0.003467
4,random_forest,Random Forest,0.898305,0.886818,0.892981,0.889889,0.902844,0.097156,0.107019,0.960521,0.958945,4.986415,0.273513
5,decision_tree,Decision Tree,0.860889,0.842896,0.857540,0.850155,0.863744,0.136256,0.142460,0.895800,0.871429,0.145632,0.001425
6,naive_bayes,Naive Bayes,0.720819,0.944969,0.417651,0.579277,0.979265,0.020735,0.582349,0.908283,0.889840,0.007352,0.003138


## Why not accuracy alone

In [3]:
ax = comparison.set_index('display_name')[['accuracy', 'f1_score', 'false_positive_rate']].plot(
    kind='bar', figsize=(9, 5))
ax.set_ylabel('Score')
ax.set_title('Accuracy vs F1 vs false positive rate')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

Naive Bayes typically shows the gap clearly: high precision and a very low false positive rate, but much lower recall, so its accuracy and F1 disagree noticeably - the reason the project does not rank models on accuracy alone.

## Cross validation and the H0 / H1 hypothesis test

In [4]:
from app.ml.cross_validation import cross_validate_all, friedman_test
from app.ml.preprocessing import get_processed_dataset, split_features_labels
from app.feature_engineering.extractor import FEATURE_NAMES
from app.ml.registry import MODEL_NAMES

frame = get_processed_dataset()
features, labels = split_features_labels(frame, FEATURE_NAMES)
cv_results = cross_validate_all(features, labels, MODEL_NAMES, folds=5, scoring='f1')
friedman_test(cv_results)

2026-09-26 13:13:00 | INFO     | app.ml.cross_validation | Logistic Regression cross validation f1: 0.8962 (+/- 0.0047)


2026-09-26 13:13:00 | INFO     | app.ml.cross_validation | Naive Bayes cross validation f1: 0.5508 (+/- 0.0070)


2026-09-26 13:13:01 | INFO     | app.ml.cross_validation | Decision Tree cross validation f1: 0.8537 (+/- 0.0081)


2026-09-26 13:13:26 | INFO     | app.ml.cross_validation | Random Forest cross validation f1: 0.8845 (+/- 0.0062)


2026-09-26 13:14:59 | INFO     | app.ml.cross_validation | SVM cross validation f1: 0.8949 (+/- 0.0050)


2026-09-26 13:15:06 | INFO     | app.ml.cross_validation | XGBoost cross validation f1: 0.8921 (+/- 0.0069)


2026-09-26 13:15:10 | INFO     | app.ml.cross_validation | ANN cross validation f1: 0.8961 (+/- 0.0054)


{'test': 'friedman',
 'performed': True,
 'models': ['logistic_regression',
  'naive_bayes',
  'decision_tree',
  'random_forest',
  'svm',
  'xgboost',
  'ann'],
 'statistic': 26.742857142857133,
 'p_value': 0.0001617818891405062,
 'alpha': 0.05,
 'reject_null_hypothesis': True,
 'interpretation': 'H0 is rejected: the algorithms differ significantly.'}